In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
import joblib
import plotly.graph_objects as go
import os

In [2]:
df = pd.read_csv("/content/clean_data.csv")
df['ds'] = pd.to_datetime(df['ds'])

# Konvertera datum till antal dagar sedan start
df['days'] = (df['ds'] - df['ds'].min()).dt.days

print("Shape:", df.shape)
print(df.head())

Shape: (7428, 3)
          ds     y  days
0 1996-04-02 -1.00     0
1 1996-04-03  2.70     1
2 1996-04-04  3.00     2
3 1996-04-05  2.10     3
4 1996-04-06  4.65     4


In [3]:
X = df[['days']]
y = df['y']

model = LinearRegression()
model.fit(X, y)

print(f"Trend: {model.coef_[0]:.6f}°C per dag")
print(f"Intercept: {model.intercept_:.2f}")

Trend: 0.000050°C per dag
Intercept: 9.15


In [4]:
# Skapa framtida datum
last_date = df['ds'].max()
future_dates = pd.date_range(start=last_date + pd.Timedelta(days=1), periods=1461, freq='D')
future_days = (future_dates - df['ds'].min()).days.values.reshape(-1, 1)

future_pred = model.predict(future_days)

# Beräkna felmarginal från residualer
residuals = y - model.predict(X)
margin = 1.96 * residuals.std()

future_df = pd.DataFrame({
    'ds': future_dates,
    'yhat': future_pred,
    'yhat_lower': future_pred - margin,
    'yhat_upper': future_pred + margin
})

print(f"Felmarginal: ±{margin:.2f}°C")
print(future_df.head())

Felmarginal: ±13.07°C
          ds      yhat  yhat_lower  yhat_upper
0 2026-02-01  9.687206   -3.380465   22.754876
1 2026-02-02  9.687255   -3.380415   22.754925
2 2026-02-03  9.687305   -3.380365   22.754975
3 2026-02-04  9.687354   -3.380316   22.755024
4 2026-02-05  9.687404   -3.380266   22.755074


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


In [5]:
mae = mean_absolute_error(y, model.predict(X))
rmse = np.sqrt(mean_squared_error(y, model.predict(X)))

print(f"MAE:  {mae:.2f}°C")
print(f"RMSE: {rmse:.2f}°C")

MAE:  5.71°C
RMSE: 6.67°C


In [6]:
future_df['year'] = future_df['ds'].dt.year

summary = future_df.groupby('year')[['yhat', 'yhat_lower', 'yhat_upper']].mean().round(2)
print(summary)

      yhat  yhat_lower  yhat_upper
year                              
2026  9.70       -3.37       22.76
2027  9.71       -3.35       22.78
2028  9.73       -3.34       22.80
2029  9.75       -3.32       22.82
2030  9.76       -3.31       22.83


In [7]:
os.makedirs("../models", exist_ok=True)
joblib.dump(model, "../models/regression_model.pkl")
print("Modell sparad!")

Modell sparad!
